# Rocket Landing: Multi-Scale Fault-Aware Model

This notebook trains a neural network to predict optimal ignition altitudes across a vast range of rocket scales—from 500g model rockets to 15kg experimental vehicles.

### Scales Covered:
1. **Small (0.5kg)**: Low inertia, very fast burn, highly wind-susceptible.
2. **Medium (1.5kg)**: High Power Rocketry (HPR) standards.
3. **Large (10kg+)**: Ultra-high power heavy-lifters with long burns.

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import random
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, trapezoid
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm
from datetime import datetime
import pickle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
class PhysicsEngine:
    def __init__(self, config):
        self.config = config
        self.g = config.get('gravity', 9.81)
        self.rho_0 = config.get('air_density', 1.225)
        self.Cd = config.get('drag_coefficient', 0.5)
        self.A_ref = config.get('reference_area', 0.07068)
        self.wind_speed = config.get('wind_speed', 0.0)
        self.wind_direction = np.radians(config.get('wind_direction', 0.0))
        
    def get_air_density(self, altitude):
        H = 8500
        return self.rho_0 * np.exp(-max(0, altitude) / H)
    
    def get_drag_force(self, velocity, position, time):
        rho = self.get_air_density(position[2])
        wind = np.array([self.wind_speed * np.cos(self.wind_direction), self.wind_speed * np.sin(self.wind_direction), 0])
        v_rel = velocity - wind
        v_mag = np.linalg.norm(v_rel)
        if v_mag < 0.01: return np.zeros(3)
        return -0.5 * rho * v_mag**2 * self.Cd * self.A_ref * (v_rel / v_mag)

    def quaternion_to_rotation_matrix(self, q):
        w, x, y, z = q
        return np.array([
            [1 - 2*(y**2 + z**2), 2*(x*y - w*z), 2*(x*z + w*y)],
            [2*(x*y + w*z), 1 - 2*(x**2 + z**2), 2*(y*z - w*x)],
            [2*(x*z - w*y), 2*(y*z + w*x), 1 - 2*(x**2 + y**2)]
        ])

    def normalize_quaternion(self, q):
        norm = np.linalg.norm(q)
        return q / norm if norm > 1e-10 else np.array([1,0,0,0])

    def quaternion_multiply(self, q1, q2):
        w1, x1, y1, z1 = q1; w2, x2, y2, z2 = q2
        return np.array([w1*w2 - x1*x2 - y1*y2 - z1*z2, w1*x2 + x1*w2 + y1*z2 - z1*y2, w1*y2 - x1*z2 + y1*w2 + z1*x2, w1*z2 + x1*y2 - y1*x2 + z1*w2])

class SolidMotor:
    def __init__(self, config):
        tc = config.get('thrust_curve', [[0, 0], [0.1, 1500], [4.0, 1500], [4.1, 0]])
        self.thrust_interp = interp1d([t for t, _ in tc], [f for _, f in tc], bounds_error=False, fill_value=0.0)
        self.burn_time = max([t for t, _ in tc]); self.propellant_mass = config.get('propellant_mass', 10.0)
        self.mass_flow_rate = self.propellant_mass / self.burn_time
        self.total_impulse = trapezoid([f for _, f in tc], [t for t, _ in tc])
        self.tvc_max = np.radians(config.get('tvc_max_angle', 5.0))
        self.ignited = False; self.ign_time = 0; self.cur_tvc = np.zeros(2); self.cmd_tvc = np.zeros(2)

    def ignite(self, t): self.ignited = True; self.ign_time = t
    def get_thrust(self, t): return self.thrust_interp(t - self.ign_time) if self.ignited and 0 <= t - self.ign_time <= self.burn_time else 0.0
    def set_tvc_command(self, p, y): self.cmd_tvc = np.clip([p, y], -self.tvc_max, self.tvc_max)
    def update_tvc(self, dt): self.cur_tvc += (self.cmd_tvc - self.cur_tvc) * dt / 0.1
    def get_thrust_vector(self, t, R): 
        mag = self.get_thrust(t); p, y = self.cur_tvc
        return R @ (mag * np.array([np.sin(p), -np.sin(y), np.cos(p)*np.cos(y)]))

class StateEstimator:
    def __init__(self, m0, cd0):
        self.x = np.array([m0, cd0]); self.P = np.diag([2.0, 0.05]); self.Q = np.diag([0.05, 0.005]); self.R = 0.5
    def predict(self, m_dot): self.x[0] -= m_dot * 0.01; self.P += self.Q
    def update(self, a_z, v_z, rho, thr_z, A_ref):
        m, cd = self.x; qf = -0.5 * rho * v_z * abs(v_z) * A_ref; h = (thr_z + qf*cd)/m
        H = np.array([-h/m, qf/m]); S = H @ self.P @ H.T + self.R; K = self.P @ H.T / S
        self.x += K * (a_z - h); self.P = (np.eye(2) - np.outer(K, H)) @ self.P
        self.x = np.clip(self.x, [0.1, 0.1], [250.0, 2.5])
        return self.x

class FaultInjector:
    def __init__(self, config):
        self.events = []
        for ft in ['mass_drop', 'drag_change', 'thrust_anomaly']:
            if random.random() < config.get(f'{ft}_prob', 0.0):
                t = random.uniform(2.0, 8.0)
                v = random.uniform(config.get(f'{ft}_min', 1.1), config.get(f'{ft}_max', 2.0))
                self.events.append({'type': ft, 'time': t, 'val': v})
        self.events.sort(key=lambda x: x['time'])

In [ ]:
class SuicideBurnSimulation:
    def __init__(self, r_cfg, e_cfg, s_cfg):
        self.r_cfg = r_cfg; self.e_cfg = e_cfg; self.s_cfg = s_cfg
        self.physics = PhysicsEngine(e_cfg); self.motor = SolidMotor(r_cfg)
        self.dry_mass = r_cfg['dry_mass']; self.prop_mass = r_cfg['propellant_mass']; self.A_ref = e_cfg['reference_area']
        self.d_mult = 1.0; self.t_mult = 1.0
        
    def calculate_ignition_altitude(self, vz, z):
        m0 = self.dry_mass + self.prop_mass; thrust = self.motor.total_impulse / self.motor.burn_time
        a = (thrust / m0) - self.physics.g
        if a <= 0: return z
        return max(2.5, (vz**2) / (2 * a))

    def state_derivative(self, t, y):
        pos, vel, q, omg, m = y[0:3], y[3:6], y[6:10], y[10:13], y[13]
        R = self.physics.quaternion_to_rotation_matrix(q)
        F_g = np.array([0, 0, -m*self.physics.g]); F_d = self.physics.get_drag_force(vel, pos, t) * self.d_mult; F_t = self.motor.get_thrust_vector(t, R) * self.t_mult
        acc = (F_g + F_d + F_t) / m; dq = 0.5 * self.physics.quaternion_multiply(q, [0, *omg])
        dm = -self.motor.get_thrust(t) * self.motor.mass_flow_rate / (self.motor.total_impulse / self.motor.burn_time)
        return np.concatenate([vel, acc, dq, -0.1*omg, [dm]])

    def run_simulation(self, init_state, ign_alt=None, fixed_params=None):
        self.motor = SolidMotor(self.r_cfg)
        if fixed_params: 
            self.d_mult = fixed_params.get('drag_multiplier', 1.0); self.t_mult = fixed_params.get('thrust_multiplier', 1.0); self.dry_mass = fixed_params.get('dry_mass', self.dry_mass); self.faults = None
        else: 
            self.d_mult = 1.0; self.t_mult = 1.0; self.faults = FaultInjector(self.s_cfg.get('faults', {}))
            
        est = StateEstimator(init_state[13], self.physics.Cd); cur_y = init_state.copy(); t_offset = 0; landed = False; is_pow = False
        hist = {'t':[], 'z':[], 'vz':[], 'mass':[], 'inf_m':[], 'inf_cd':[], 'faults':[], 'y_full':[]}
        if ign_alt is None: ign_alt = self.calculate_ignition_altitude(cur_y[5], cur_y[2])
        while t_offset < 60 and not landed:
            t_target = 60; pend = []
            if self.faults: pend = [f for f in self.faults.events if f['time'] > t_offset];
            if pend: t_target = pend[0]['time']
            def gnd(t, y): return y[2]
            gnd.terminal = True; gnd.direction = -1
            def ign(t, y): return y[2] - ign_alt if not is_pow else 1.0
            ign.terminal = True; ign.direction = -1
            sol = solve_ivp(self.state_derivative, [t_offset, t_target], cur_y, events=[gnd, ign], rtol=1e-5)
            for i in range(len(sol.t)):
                pt_y = sol.y[:, i]; pt_t = sol.t[i]; R = self.physics.quaternion_to_rotation_matrix(pt_y[6:10])
                thr_b = R.T @ (self.motor.get_thrust_vector(pt_t, R)*self.t_mult); acc_z = (self.state_derivative(pt_t, pt_y)[5] + self.physics.g)
                est_state = est.update(acc_z, pt_y[5], self.physics.get_air_density(pt_y[2]), thr_b[2], self.A_ref); est.predict(0.1)
                hist['t'].append(pt_t); hist['z'].append(pt_y[2]); hist['vz'].append(pt_y[5]); hist['mass'].append(pt_y[13]); hist['inf_m'].append(est_state[0]); hist['inf_cd'].append(est_state[1]); hist['y_full'].append(pt_y)
            cur_y = sol.y[:, -1]; t_offset = sol.t[-1]
            if len(sol.t_events[0]) > 0: landed = True
            if not is_pow and len(sol.t_events) > 1 and len(sol.t_events[1]) > 0: is_pow = True; self.motor.ignite(t_offset)
            if self.faults and pend and abs(t_offset - t_target) < 1e-3:
                f = pend[0]; hist['faults'].append(f)
                if f['type'] == 'mass_drop': cur_y[13] -= f['val']
                elif f['type'] == 'drag_change': self.d_mult = f['val']
                elif f['type'] == 'thrust_anomaly': self.t_mult = 1.0 / f['val']
        return abs(cur_y[5]) < 3.0, cur_y, hist
        
    def optimize_oracle(self, state, params):
        best_alt = self.calculate_ignition_altitude(state[5], state[2]); best_v = 999
        search = np.linspace(max(2.5, best_alt-20), best_alt+20, 11)
        for alt in search:
            s, final, _ = self.run_simulation(state.copy(), alt, fixed_params=params)
            if abs(final[5]) < best_v: best_v = abs(final[5]); best_alt = alt
        return best_alt

In [ ]:
def generate_data(num_flights=100):
    dataset = []
    
    # Defined Rocket Classes
    classes = [
        {'name': 'Small', 'dry_range': (0.3, 0.7), 'prop_range': (0.1, 0.2), 'dia': 0.04, 'peak': (80, 150), 'burn': (1.2, 1.8)},
        {'name': 'Medium', 'dry_range': (1.2, 3.5), 'prop_range': (0.3, 0.8), 'dia': 0.076, 'peak': (350, 700), 'burn': (3.0, 4.5)},
        {'name': 'Large', 'dry_range': (8.0, 15.0), 'prop_range': (1.5, 3.5), 'dia': 0.15, 'peak': (2000, 3500), 'burn': (5.0, 8.0)}
    ]

    print(f"Generating data across {len(classes)} rocket classes...")
    for _ in tqdm(range(num_flights)):
        cls = random.choice(classes)
        dry_mass = random.uniform(*cls['dry_range'])
        prop_mass = random.uniform(*cls['prop_range'])
        A_ref = np.pi * (cls['dia']/2)**2
        
        peak = random.uniform(*cls['peak']); burn = random.uniform(*cls['burn'])
        tc = [[0,0], [0.1, peak], [burn, peak], [burn+0.1, 0]]
        
        wind_v = random.uniform(0, 12); air_rho = random.uniform(1.1, 1.3); base_cd = random.uniform(0.4, 0.6)
        
        r_cfg = {'dry_mass': dry_mass, 'propellant_mass': prop_mass, 'thrust_curve': tc, 'tvc_max_angle': 6.0}
        e_cfg = {'gravity': 9.81, 'air_density': air_rho, 'drag_coefficient': base_cd, 'reference_area': A_ref, 'wind_speed': wind_v, 'wind_direction': 0}
        s_cfg = {'faults': {'mass_drop_prob': 0.4, 'drag_change_prob': 0.4, 'thrust_anomaly_prob': 0.4, 'mass_drop_min': 0.1, 'mass_drop_max': 2.0}}
        
        sim = SuicideBurnSimulation(r_cfg, e_cfg, s_cfg)
        h0 = random.uniform(250, 500); v0 = random.uniform(-15, -60)
        init_state = np.array([0,0,h0, 0,0,v0, 1,0,0,0, 0,0,0, dry_mass + prop_mass])
        
        success, _, hist = sim.run_simulation(init_state)
        ascent_twr = peak / ((dry_mass + prop_mass) * 9.81)
        
        for i in range(15, len(hist['t']), 12):
            if hist['z'][i] < 30 or hist['vz'][i] > -5: continue
            
            params = {'drag_multiplier': 1.0, 'thrust_multiplier': 1.0, 'dry_mass': dry_mass}
            for f in hist['faults']:
                if f['time'] <= hist['t'][i]:
                    if f['type'] == 'drag_change': params['drag_multiplier'] = f['val']
                    elif f['type'] == 'thrust_anomaly': params['thrust_multiplier'] = 1.0/f['val']
                    elif f['type'] == 'mass_drop': params['dry_mass'] -= f['val']
            
            opt_alt = sim.optimize_oracle(hist['y_full'][i], params)
            dataset.append({
                'ascent_twr': ascent_twr, 'descent_velocity': hist['vz'][i], 'current_altitude': hist['z'][i],
                'inferred_mass': hist['inf_m'][i], 'inferred_drag_coeff': hist['inf_cd'][i], 'ambient_temp': 288.15,
                'wind_speed': wind_v, 'Predicted_ignition_altitude': sim.calculate_ignition_altitude(hist['vz'][i], hist['z'][i]),
                'TARGET_optimal_ignition_altitude': opt_alt
            })
            
    df = pd.DataFrame(dataset)
    df.to_csv('training_data.csv', index=False)
    return df

In [ ]:
df = generate_data(num_flights=250)
print(f"Database populated with {len(df)} samples across all scales.")

In [ ]:
feature_cols = ['ascent_twr', 'descent_velocity', 'current_altitude', 'inferred_mass', 'inferred_drag_coeff', 'ambient_temp', 'wind_speed', 'Predicted_ignition_altitude']
X = df[feature_cols].values; y = df['TARGET_optimal_ignition_altitude'].values
scaler = StandardScaler(); X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.15, random_state=42)
with open('scaler.pkl', 'wb') as f: pickle.dump(scaler, f)

model = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=[len(feature_cols)]),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])
model.compile(loss='huber', optimizer=tf.keras.optimizers.Adam(0.0005), metrics=['mae'])
history = model.fit(X_train, y_train, validation_split=0.2, epochs=250, batch_size=64, verbose=0)
print(f"Final Accuracy (MAE): {model.evaluate(X_test, y_test)[1]:.2f}m")
model.save('ignition_model.keras')